## 1. Setup & Imports

In [ ]:
# If running in Google Colab, you may need to install some libraries.
# Uncomment if required.
# !pip install pandas numpy matplotlib seaborn scikit-learn ta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure(figsize)'] = (8, 4)
sns.set_style('whitegrid')


## 2. Problem Statement & Objectives

**Business Problem**  
Cryptocurrency markets are highly volatile. We want to build a model that can **predict future volatility levels** using historical OHLC (Open, High, Low, Close) prices, trading volume, and market capitalization.

**Goals:**
- Predict short‑term volatility of a given cryptocurrency (regression task).
- Help traders/institutions anticipate high‑risk periods.
- Build a clean, reusable ML pipeline.

**Dataset Structure (expected):**
- `date` – trading day
- `symbol` – cryptocurrency ticker (e.g., BTC, ETH)
- `open`, `high`, `low`, `close`
- `volume`
- `market_cap`

You may have additional columns, which can be used for feature engineering.

## 3. Data Loading

In [ ]:
# Option 1: Load CSV from local upload in Colab
from google.colab import files

print('Please upload your cryptocurrency historical prices CSV file...')
uploaded = files.upload()  # This will open a file chooser in Colab

csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)

print('Shape:', df.shape)
df.head()

## 4. Initial Data Inspection

In [ ]:
# Basic overview
df.info()


In [ ]:
# Preview statistics
df.describe(include='all').T

In [ ]:
# Check missing values
df.isna().sum()

## 5. Data Cleaning & Preprocessing

Steps:
1. Convert `date` to datetime & sort.
2. Handle missing values (forward fill / drop / interpolation).
3. Filter for a single symbol (or loop later if multi‑asset).
4. Remove obvious outliers if necessary.

In [ ]:
# Convert date column and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

# Example: focus on one symbol for a start (e.g., BTC). Change as needed.
symbol_to_use = df['symbol'].unique()[0]
coin_df = df[df['symbol'] == symbol_to_use].copy().reset_index(drop=True)

print('Working on symbol:', symbol_to_use)
coin_df.head()

In [ ]:
# Handle missing values – simple strategy
coin_df = coin_df.fillna(method='ffill').fillna(method='bfill')
coin_df.isna().sum()

## 6. Feature Engineering – Volatility & Technical Indicators

We will create:
- Daily returns
- Rolling volatility (e.g., 7‑day std of returns)
- Moving averages (MA7, MA14, MA30)
- Liquidity ratio: `volume / market_cap`

These engineered features become inputs to our ML model. The **target variable** here will be the *future volatility*, for example the rolling volatility shifted one day ahead.

In [ ]:
# Calculate returns
coin_df['return'] = coin_df['close'].pct_change()

# Rolling volatility (e.g., 7‑day)
window_vol = 7
coin_df['rolling_volatility'] = coin_df['return'].rolling(window_vol).std()

# Moving averages
coin_df['ma7'] = coin_df['close'].rolling(7).mean()
coin_df['ma14'] = coin_df['close'].rolling(14).mean()
coin_df['ma30'] = coin_df['close'].rolling(30).mean()

# Liquidity ratio
coin_df['liquidity_ratio'] = coin_df['volume'] / coin_df['market_cap']

# Drop initial rows with NaNs caused by rolling calculations
coin_df = coin_df.dropna().reset_index(drop=True)

coin_df[['date', 'close', 'return', 'rolling_volatility', 'ma7', 'liquidity_ratio']].head()

## 7. Exploratory Data Analysis (EDA)

In [ ]:
# Price & volatility over time
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(coin_df['date'], coin_df['close'])
ax1.set_xlabel('Date')
ax1.set_ylabel('Close Price')
ax1.set_title(f'{symbol_to_use} – Close Price Over Time')
plt.show()

fig, ax2 = plt.subplots(figsize=(10,4))
ax2.plot(coin_df['date'], coin_df['rolling_volatility'])
ax2.set_xlabel('Date')
ax2.set_ylabel('Rolling Volatility')
ax2.set_title(f'{symbol_to_use} – Rolling Volatility ({window_vol}‑day)')
plt.show()

In [ ]:
# Correlation heatmap for engineered features
feature_cols_for_corr = ['open', 'high', 'low', 'close', 'volume', 'market_cap',
                         'return', 'rolling_volatility', 'ma7', 'ma14', 'ma30', 'liquidity_ratio']

corr = coin_df[feature_cols_for_corr].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## 8. Train–Test Split & Scaling

Target variable: we use **future rolling volatility** (next day) as the prediction target:

```python
coin_df['target_volatility'] = coin_df['rolling_volatility'].shift(-1)
```

We then drop the last row (its target is NaN) and split into train/test sets in time order.

In [ ]:
# Define target as next‑day volatility
coin_df['target_volatility'] = coin_df['rolling_volatility'].shift(-1)
coin_df = coin_df.dropna().reset_index(drop=True)

feature_cols = ['open', 'high', 'low', 'close', 'volume', 'market_cap',
                'return', 'rolling_volatility', 'ma7', 'ma14', 'ma30', 'liquidity_ratio']

X = coin_df[feature_cols].values
y = coin_df['target_volatility'].values

# Time‑based split (e.g., 80% train, 20% test)
split_idx = int(len(coin_df) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

## 9. Baseline Models (Linear Regression & Random Forest)

In [ ]:
def evaluate_regression_model(model, X_tr, y_tr, X_te, y_te, model_name="Model"):
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, y_pred_te))
    mae_te = mean_absolute_error(y_te, y_pred_te)
    r2_te = r2_score(y_te, y_pred_te)

    print(f"\n==== {model_name} ====")
    print(f"Train RMSE: {rmse_tr:.6f}")
    print(f"Test  RMSE: {rmse_te:.6f}")
    print(f"Test  MAE : {mae_te:.6f}")
    print(f"Test  R^2 : {r2_te:.4f}")

    return model

# Linear Regression
lin_reg = LinearRegression()
lin_reg = evaluate_regression_model(lin_reg, X_train_scaled, y_train, X_test_scaled, y_test, 'Linear Regression')

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg = evaluate_regression_model(rf_reg, X_train_scaled, y_train, X_test_scaled, y_test, 'Random Forest Regressor')

## 10. Hyperparameter Tuning (Example with Random Forest)

In [ ]:
# Example grid search – be careful with time in Colab
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

tscv = TimeSeriesSplit(n_splits=3)
grid_search = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                           param_grid,
                           cv=tscv,
                           scoring='neg_mean_squared_error',
                           n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)
print('Best Params:', grid_search.best_params_)

best_rf = grid_search.best_estimator_
best_rf = evaluate_regression_model(best_rf, X_train_scaled, y_train, X_test_scaled, y_test, 'Best RF (GridSearch)')

## 11. Save Trained Model & Scaler

In [ ]:
import joblib

joblib.dump(best_rf, 'crypto_volatility_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')
print('Saved model and scaler as .pkl files.')

## 12. Simple Inference Function

In [ ]:
def predict_future_volatility(latest_row: pd.Series):
    """Given a row with feature columns, predict next‑day volatility."""
    model = joblib.load('crypto_volatility_model.pkl')
    scaler = joblib.load('feature_scaler.pkl')

    x = latest_row[feature_cols].values.reshape(1, -1)
    x_scaled = scaler.transform(x)
    pred_vol = model.predict(x_scaled)[0]
    return pred_vol

# Example usage (using last available row)
latest_row = coin_df.iloc[-1]
predicted_vol = predict_future_volatility(latest_row)
print('Predicted next‑day volatility:', predicted_vol)

## 13. Conclusion & Next Steps

In this notebook we:
- Loaded and cleaned historical cryptocurrency OHLC data.
- Engineered meaningful features like returns, rolling volatility, moving averages, and liquidity ratios.
- Performed EDA to understand price and volatility behavior.
- Built and evaluated baseline ML models to predict **future volatility**.
- Tuned a Random Forest model and saved it for reuse.
